# WHOOP Data Cleaning Pipeline

*Generated 2025-04-26*

This notebook replicates the automated cleaning you just tested. It turns raw exports produced by `fetch_whoop_data.py` into two tidy files:

1. **`clean_daily.csv`** – day‑level aggregate (Cycles + Sleep + Recovery)
2. **`clean_workout.csv`** – trimmed workout log with intuitive column names and sport labels

Feel free to adapt paths or add extra features.

## 1 · Setup – imports & paths

In [ ]:
import pandas as pd
from pathlib import Path

RAW_DIR = Path('whoop_exports')  # folder with your raw CSVs
OUT_DIR = RAW_DIR                # feel free to change


## 2 · Load raw exports

In [ ]:
cycles   = pd.read_csv(next(RAW_DIR.glob('cycles_*.csv')))
sleep    = pd.read_csv(next(RAW_DIR.glob('sleep_*.csv')))
recovery = pd.read_csv(next(RAW_DIR.glob('recovery_*.csv')))
workout  = pd.read_csv(next(RAW_DIR.glob('workout_*.csv')))

print('Loaded:', len(cycles), 'cycles', '/', len(workout), 'workouts')


## 3 · Drop empty rows / columns

In [ ]:
def drop_empty(df):
    df = df.dropna(how='all')
    return df.loc[:, ~df.isna().all()]

cycles   = drop_empty(cycles)
sleep    = drop_empty(sleep)
recovery = drop_empty(recovery)
workout  = drop_empty(workout)


## 4 · Clean workout table

In [ ]:
# Columns to keep
keep_cols = [
    'user_id','id','start','end','sport_id',
    'score.strain','score.average_heart_rate','score.max_heart_rate','score.kilojoule',
    'score.zone_duration.zone_zero_milli','score.zone_duration.zone_one_milli',
    'score.zone_duration.zone_two_milli','score.zone_duration.zone_three_milli',
    'score.zone_duration.zone_four_milli','score.zone_duration.zone_five_milli'
]
workout = workout[keep_cols]

# Rename columns
workout = workout.rename(columns={
    'id':'workout_id',
    'sport_id':'sport',
    'score.strain':'strain',
    'score.average_heart_rate':'avg_hr',
    'score.max_heart_rate':'max_hr',
    'score.kilojoule':'kilojoule',
    'score.zone_duration.zone_zero_milli':'zone0_ms',
    'score.zone_duration.zone_one_milli':'zone1_ms',
    'score.zone_duration.zone_two_milli':'zone2_ms',
    'score.zone_duration.zone_three_milli':'zone3_ms',
    'score.zone_duration.zone_four_milli':'zone4_ms',
    'score.zone_duration.zone_five_milli':'zone5_ms'
})


### Map sport IDs → names

In [ ]:
sport_map = {
 -1:'Activity',0:'Running',1:'Cycling',16:'Baseball',17:'Basketball',18:'Rowing',
 19:'Fencing',20:'Field Hockey',21:'Football',22:'Golf',24:'Ice Hockey',
 25:'Lacrosse',27:'Rugby',28:'Sailing',29:'Skiing',30:'Soccer',31:'Softball',
 32:'Squash',33:'Swimming',34:'Tennis',35:'Track & Field',36:'Volleyball',
 37:'Water Polo',38:'Wrestling',39:'Boxing',42:'Dance',43:'Pilates',44:'Yoga',
 45:'Weightlifting',47:'Cross Country Skiing',48:'Functional Fitness',
 49:'Duathlon',51:'Gymnastics',52:'Hiking/Rucking',53:'Horseback Riding',
 55:'Kayaking',56:'Martial Arts',57:'Mountain Biking',59:'Powerlifting',
 60:'Rock Climbing',61:'Paddleboarding',62:'Triathlon',63:'Walking',
 64:'Surfing',65:'Elliptical',66:'Stairmaster',70:'Meditation',71:'Other',
 73:'Diving',74:'Operations - Tactical',75:'Operations - Medical',
 76:'Operations - Flying',77:'Operations - Water',82:'Ultimate',83:'Climber',
 84:'Jumping Rope',85:'Australian Football',86:'Skateboarding',87:'Coaching',
 88:'Ice Bath',89:'Commuting',90:'Gaming',91:'Snowboarding',92:'Motocross',
 93:'Caddying',94:'Obstacle Course Racing',95:'Motor Racing',96:'HIIT',
 97:'Spin',98:'Jiu Jitsu',99:'Manual Labor',100:'Cricket',101:'Pickleball',
 102:'Inline Skating',103:'Box Fitness',104:'Spikeball',105:'Wheelchair Pushing',
 106:'Paddle Tennis',107:'Barre',108:'Stage Performance',109:'High Stress Work',
 110:'Parkour',111:'Gaelic Football',112:'Hurling/Camogie',113:'Circus Arts',
 121:'Massage Therapy',123:'Strength Trainer',125:'Watching Sports',
 126:'Assault Bike',127:'Kickboxing',128:'Stretching',230:'Table Tennis',
 231:'Badminton',232:'Netball',233:'Sauna',234:'Disc Golf',235:'Yard Work',
 236:'Air Compression',237:'Percussive Massage',238:'Paintball',239:'Ice Skating',
 240:'Handball',248:'F45 Training',249:'Padel',250:"Barry's",251:'Dedicated Parenting',
 252:'Stroller Walking',253:'Stroller Jogging',254:'Toddlerwearing',255:'Babywearing',
 258:'Barre3',259:'Hot Yoga',261:'Stadium Steps',262:'Polo',263:'Musical Performance',
 264:'Kite Boarding',266:'Dog Walking',267:'Water Skiing',268:'Wakeboarding',
 269:'Cooking',270:'Cleaning',272:'Public Speaking'
}


workout['sport'] = workout['sport'].map(sport_map).fillna('Unknown')
workout.head()

## 5 · Build clean daily table

In [ ]:
# Align key names
sleep = sleep.rename(columns={'cycle_id':'id'})
recovery = recovery.rename(columns={'cycle_id':'id'})

# Merge
daily = cycles.merge(sleep, on='id', how='left', suffixes=('','_sleep'))
daily = daily.merge(recovery, on='id', how='left', suffixes=('','_rec'))

# Drop all-null cols
daily = daily.loc[:, ~daily.isna().all()]

# Rename some common metrics for readability (optional, extend to taste)
rename_map = {
    'score.strain':'strain',
    'score.average_heart_rate':'avg_hr',
    'score.kilojoule':'kilojoule',
    'score.recovery_score':'recovery_score',
    'score.resting_heart_rate':'rhr',
    'score.hrv_rmssd_milli':'hrv_ms',
    'score.performance_percentage':'sleep_perf_pct',
    'score.stage_summary.total_in_bed_time_milli':'time_in_bed_ms',
    'score.stage_summary.total_sleep_time_milli':'sleep_ms'
}
daily = daily.rename(columns={k:v for k,v in rename_map.items() if k in daily.columns})

daily.head()


## 6 · Save cleaned CSVs

In [ ]:
OUT_DIR.mkdir(exist_ok=True)
daily_path   = OUT_DIR / 'clean_daily.csv'
workout_path = OUT_DIR / 'clean_workout.csv'

daily.to_csv(daily_path, index=False)
workout.to_csv(workout_path, index=False)

print('Wrote', daily_path, 'and', workout_path)
